# 08 生成模块与提示词工程第二部分：Ollama 生成 + 端到端流水线

> 本 notebook **贯穿阶段 0–6**，随开发进度增量追加 C0→C7（见 [`schedule.md`](../schedule.md)「开发与备份工作流」）。
>
> **每完成一个开发阶段**：运行本阶段对应 cell → 更新 `schedule.md` + 根目录 `README.md` → git 提交 → 再进入下一阶段编码。

---

## 章节与阶段对照（增量追加）

| 章节 | 开发阶段 | 内容 | 状态 |
|------|----------|------|------|
| **C0** | 0 环境与骨架 | 路径、`sys.path`、上游 import、Ollama 配置 | ✅ 已完成 |
| **C1** | 1 LLMGenerator | `health_check` + 单条 `generate()` | ✅ 已完成 |
| C2 | 2 JSON 工具 | `extract_json` / `repair_json` | 🔄 下一步 |
| C3 | 3 Pipeline | 06→07 联调 | ☐ |
| C4 | 3 Pipeline | 最小路径 `run()` | ☐ |
| C5 | 3 Pipeline | 完整 pipeline + optional stages | ☐ |
| C6 | 4 后处理 | `sources`、引用、免责声明 | ☐ |
| C7 | 5–6 评测交付 | 批量 query + `generation_eval.json` | ☐ |

> 内核 **`med-rag-verify`**。Ollama 日常启动见 **C0**；**安装 / 拉模型 / PATH 排错** 见 [`schedule.md`](../schedule.md)「Windows Ollama 环境（方式 A）」。

## C0：环境与路径（阶段 0）

1. 解析路径并 smoke import 上游 06/07；
2. 探测 Ollama 是否在线且已有 `deepseek-r1:7b`。

**日常启动 Ollama（方式 A）**：开始菜单或托盘打开 **Ollama** → 运行下方两个 code cell。若探测失败，见 [`schedule.md` § Windows Ollama 环境](../schedule.md)。

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

# 确保可 import 本阶段 bootstrap（notebook 位于 08/notebooks）
_nb_dir = Path.cwd().resolve()
if _nb_dir.name == "notebooks":
    _stage08_src = _nb_dir.parent / "src"
else:
    _stage08_src = _nb_dir / "08 生成模块与提示词工程第二部分" / "src"
if str(_stage08_src) not in sys.path:
    sys.path.insert(0, str(_stage08_src))

from bootstrap import OLLAMA_BASE_URL, OLLAMA_MODEL, bootstrap_paths

paths = bootstrap_paths(_nb_dir)
ROOT = paths["root"]
STAGE08 = paths["stage08"]
STAGE06 = paths["stage06"]
STAGE07 = paths["stage07"]

print("ROOT:", ROOT)
print("STAGE08:", STAGE08)
print("Ollama:", OLLAMA_BASE_URL, "| model:", OLLAMA_MODEL)

# 上游 smoke import（05 由 06 pipeline 按需挂载，此处不加入 sys.path）
from pipeline import RetrievalPipeline  # noqa: E402  06
from context_assembler import ContextAssembler  # noqa: E402  07
from prompts import PROMPT_STAGES  # noqa: E402  07

print("RetrievalPipeline:", RetrievalPipeline)
print("ContextAssembler:", ContextAssembler)
print("PROMPT_STAGES keys:", list(PROMPT_STAGES.keys()))
print("\n✅ C0 smoke: paths + upstream imports OK")

ROOT: D:\谷歌
STAGE08: D:\谷歌\08 生成模块与提示词工程第二部分
Ollama: http://127.0.0.1:11434 | model: deepseek-r1:7b
RetrievalPipeline: <class 'pipeline.RetrievalPipeline'>
ContextAssembler: <class 'context_assembler.ContextAssembler'>
PROMPT_STAGES keys: ['evidence_evaluator', 'answer_generator', 'critical_reviewer', 'final_assembler']

✅ C0 smoke: paths + upstream imports OK


### C0（续）：Ollama 探测

期望 `Probe OK: True`。失败时见 [`schedule.md`](../schedule.md)「Windows Ollama 环境」。

In [2]:
import httpx

def probe_ollama(base_url: str, model: str) -> dict:
    try:
        with httpx.Client(timeout=5.0) as client:
            resp = client.get(f"{base_url}/api/tags")
            resp.raise_for_status()
            names = [m.get("name", "") for m in resp.json().get("models", [])]
    except httpx.HTTPError as exc:
        return {"ok": False, "error": str(exc), "models": [], "reachable": False}
    has_model = any(model in n for n in names)
    return {
        "ok": has_model,
        "reachable": True,
        "models": names,
        "error": None if has_model else f"model {model!r} not in /api/tags",
    }

status = probe_ollama(OLLAMA_BASE_URL, OLLAMA_MODEL)
print("Ollama URL:", OLLAMA_BASE_URL)
print("Service reachable:", status.get("reachable", False))
print("Probe OK (model ready):", status["ok"])
if status["models"]:
    print("Installed models:", status["models"])

if not status.get("reachable"):
    print("\n⚠️ 无法连接 → 托盘/开始菜单启动 Ollama，详见 schedule.md")
elif status["error"]:
    print("\nNote:", status["error"])
    print("拉模型见 schedule.md「Windows Ollama 环境」")
else:
    print(f"\n✅ C0 Ollama OK — {OLLAMA_MODEL!r} available")

Ollama URL: http://127.0.0.1:11434
Service reachable: True
Probe OK (model ready): True
Installed models: ['deepseek-r1:7b']

✅ C0 Ollama OK — 'deepseek-r1:7b' available


## C1：LLMGenerator smoke（阶段 1）

`health_check()` + 单条 `generate()`；默认 `think=False`（01 阶段结论，避免 deepseek-r1 耗尽在 thinking）。

In [ ]:
import time

from llm_generator import LLMGenerator

llm = LLMGenerator(model_name=OLLAMA_MODEL, base_url=OLLAMA_BASE_URL, timeout=180.0)

print("health_check:", llm.health_check())

t0 = time.perf_counter()
answer = llm.generate(
    "In one sentence, what is metformin used for?",
    system_prompt="You are a concise medical assistant. Answer in English.",
    temperature=0.2,
    max_tokens=512,  # deepseek-r1 需留足预算，避免全耗在 thinking
    think=False,
)
elapsed = time.perf_counter() - t0

preview = answer[:400] + ("..." if len(answer) > 400 else "")
print(f"\nElapsed: {elapsed:.1f}s | chars: {len(answer)}")
print("Response preview:\n", preview or "(empty — 尝试增大 max_tokens)")
print("\n✅ C1 LLMGenerator smoke OK" if answer.strip() else "\n⚠️ 空响应，见 schedule / 01 阶段 think 说明")

health_check: True

Elapsed: 4.7s | chars: 158
Response preview:
 Metformin is used to treat type 2 diabetes mellitus by lowering blood glucose levels through increased insulin production and enhanced utilization of glucose.

✅ C1 LLMGenerator smoke OK
